# News RAG 구현

참고: https://kangth97.tistory.com/56

## 환경
- GPU: RTX 4070 (8GB VRAM)
- CUDA: 12.x
- Python: 3.12

## 구성
1. 환경 세팅
2. 뉴스 데이터 로드
3. 텍스트 청킹
4. 임베딩 (Korean Sentence Transformer)
5. FAISS 벡터스토어
6. LLM 설정 (4-bit 양자화, 8GB VRAM 대응)
7. RAG 체인 구성
8. 질의응답

## 1. 환경 세팅

In [4]:
# GPU 확인
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("running on:", device)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

running on: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
VRAM: 8.0 GB


In [5]:
# !pip install numpy==1.26.4

In [6]:
# 필요한 패키지 설치 (설치되어 있으면 무시됨)
# faiss-gpu 는 CUDA 12.x 에서 지원 안 됨 -> faiss-cpu 사용
%pip install -q \
    faiss-cpu \
    datasets \
    langchain \
    langchain-community \
    langchain-huggingface \
    bitsandbytes \
    accelerate \
    sentence-transformers

Note: you may need to restart the kernel to use updated packages.


## 2. 뉴스 데이터 로드

In [7]:
from datasets import load_dataset

# HuggingFace 한국어 뉴스 데이터셋 로드
# 'daekeun-ml/naver-news-summarization-ko' 는 네이버 뉴스 요약 데이터셋
dataset = load_dataset("daekeun-ml/naver-news-summarization-ko", split="train[:500]")
print(f"로드된 뉴스 수: {len(dataset)}")
print("컬럼:", dataset.column_names)
print("\n샘플 데이터:")
print("제목:", dataset[0]["title"])
print("본문 일부:", dataset[0]["document"][:200])

로드된 뉴스 수: 500
컬럼: ['date', 'category', 'press', 'title', 'document', 'link', 'summary']

샘플 데이터:
제목: 추경호 중기 수출지원 총력 무역금융 40조 확대
본문 일부: 앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 


In [8]:
from langchain_core.documents import Document

documents = [
Document(
page_content=row["document"],
metadata={"title": row["title"], "source": "naver_news"}
)
for row in dataset
if row["document"]
]

print(f"Document 수: {len(documents)}")
print("\n첫 번째 Document:")
print(documents[0])

Document 수: 500

첫 번째 Document:
page_content='앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.' metadata={'title': '추경호 중기 수출지원 총력 무역금융 40조 확대', 'source': 'naver_news'}


In [9]:
!pip install langchain-text-splitters

## 3. 텍스트 청킹

In [10]:

# 변경 후
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,        # 청크 크기 (한국어 기준 적당한 크기)
    chunk_overlap=50,      # 청크 간 겹치는 문자 수
    length_function=len,
)

chunks = text_splitter.split_documents(documents)
print(f"청킹 전: {len(documents)} 문서")
print(f"청킹 후: {len(chunks)} 청크")
print(f"평균 청크 길이: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} 글자")

청킹 전: 500 문서
청킹 후: 1296 청크
평균 청크 길이: 419 글자


## 4. 임베딩 모델 설정

한국어 뉴스에 적합한 임베딩 모델 사용:
- `jhgan/ko-sroberta-multitask`: 한국어 Sentence-BERT, 경량

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cuda"},  # GPU 활용
    encode_kwargs={"normalize_embeddings": True},
)

# 임베딩 테스트
test_vec = embedding_model.embed_query("오늘 날씨가 맑습니다.")
print(f"임베딩 차원: {len(test_vec)}")

임베딩 차원: 768


## 5. FAISS 벡터스토어 구축

> `faiss-gpu`는 CUDA 12.x 에서 지원되지 않으므로 `faiss-cpu` 사용  
> (임베딩 계산은 GPU로, 인덱싱/검색은 CPU로 수행)

In [12]:
from langchain_community.vectorstores import FAISS
import os

FAISS_INDEX_PATH = "./faiss_news_index"

if os.path.exists(FAISS_INDEX_PATH):
    # 이미 만들어진 인덱스 재사용
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        embedding_model,
        allow_dangerous_deserialization=True
    )
    print("기존 FAISS 인덱스 로드 완료")
else:
    # 새로 구축 (배치 처리로 OOM 방지)
    print("FAISS 인덱스 구축 중...")
    vectorstore = FAISS.from_documents(chunks[:100], embedding_model)  # 처음 100개로 시작

    # 나머지 배치로 추가
    batch_size = 100
    for i in range(100, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        vectorstore.add_documents(batch)
        print(f"  진행: {min(i+batch_size, len(chunks))}/{len(chunks)}")

    vectorstore.save_local(FAISS_INDEX_PATH)
    print(f"FAISS 인덱스 저장 완료: {FAISS_INDEX_PATH}")

FAISS 인덱스 구축 중...
  진행: 200/1296
  진행: 300/1296
  진행: 400/1296
  진행: 500/1296
  진행: 600/1296
  진행: 700/1296
  진행: 800/1296
  진행: 900/1296
  진행: 1000/1296
  진행: 1100/1296
  진행: 1200/1296
  진행: 1296/1296
FAISS 인덱스 저장 완료: ./faiss_news_index


In [13]:
# 검색 테스트
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # 상위 3개 문서 검색
)

test_query = "주식 시장 동향"
results = retriever.invoke(test_query)
print(f"질의: '{test_query}'")
print(f"검색된 문서 수: {len(results)}")
for i, doc in enumerate(results):
    print(f"\n[{i+1}] 제목: {doc.metadata.get('title', 'N/A')}")
    print(f"    내용: {doc.page_content[:100]}...")

질의: '주식 시장 동향'
검색된 문서 수: 3

[1] 제목: 유리자산운용 유리타겟크루즈 주식혼합형 펀드 출시
    내용: ‘변동성 트레이딩 전략’이다. ‘유리타겟크루즈펀드’는 투자되는 개별 종목별로 가격범위를 정하고 변동성으로 인한 가격범위 내에서의 등락을 활용한 매매전략을 수행한다. 또한 특정 종목...

[2] 제목: 유리자산운용 유리타겟크루즈 주식혼합형 펀드 출시
    내용: 머니팁 이데일리 이은정 기자 유리자산운용은 4일 ‘유리타겟크루즈증권투자신탁 주식혼합 ’ 펀드를 출시한다고 밝혔다. 사진 유리자산운용 이 상품은 10종목 내외의 국내 저평가 우량주식...

[3] 제목: 금융당국 “증시 변동성 완화 조치”
    내용: 하루 자기주식 매수 주문 수량 한도 제한도 완화된다. 신탁취득 주식도 발행주식총수의 1% 이내에서 신탁재산 총액 범위 내로 한시적으로 확대된다. 이와 함께 금융위는 금융감독원과 한...


## 6. LLM 설정

RTX 4070 (8GB VRAM) 기준:
- 7B ~ 8B 모델은 **4-bit 양자화** 시 약 4~5GB 사용 가능
- `bitsandbytes` 라이브러리로 `load_in_4bit=True` 설정

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline

# 4-bit 양자화 설정 (8GB VRAM 대응)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# 한국어 지원 모델 (8B, 4-bit 양자화 시 ~5GB 사용)
# 다른 옵션: "MLP-KTLim/llama-3-Korean-Bllossom-8B", "beomi/Llama-3-Open-Ko-8B"
MODEL_ID = "beomi/Llama-3-Open-Ko-8B"

print(f"모델 로드 중: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",  # GPU 자동 배치
)

print(f"GPU 메모리 사용: {torch.cuda.memory_allocated(0) / 1024**3:.1f} GB")

모델 로드 중: beomi/Llama-3-Open-Ko-8B


c:\Users\SSAFY\miniforge3\envs\llm\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--beomi--Llama-3-Open-Ko-8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP

GPU 메모리 사용: 5.7 GB


In [15]:
# HuggingFace Pipeline -> LangChain LLM 래핑
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,  # 입력 프롬프트 제외하고 생성된 부분만 반환
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

Device set to use cuda:0


## 7. RAG 체인 구성

In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# RAG 프롬프트 템플릿
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""당신은 뉴스 기사를 바탕으로 질문에 답변하는 AI 어시스턴트입니다.
아래 뉴스 기사들을 참고하여 질문에 정확하게 답변하세요.
기사에 없는 내용은 "관련 기사를 찾을 수 없습니다"라고 답변하세요.

[참고 뉴스 기사]
{context}

[질문]
{question}

[답변]"""
)

def format_docs(docs):
    """검색된 문서들을 하나의 문자열로 합침"""
    return "\n\n".join(
        f"제목: {doc.metadata.get('title', 'N/A')}\n내용: {doc.page_content}"
        for doc in docs
    )

# RAG 체인: 검색 -> 프롬프트 구성 -> LLM -> 출력 파싱
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG 체인 구성 완료")

RAG 체인 구성 완료


## 8. 질의응답

In [18]:
# 질의 실행
question = "최근 경제 동향은 어떤가요?"

print(f"질문: {question}")
print("-" * 50)

answer = rag_chain.invoke(question)
print("답변:")
print(answer)

질문: 최근 경제 동향은 어떤가요?
--------------------------------------------------
답변:
그런데 그때 갑자기 들려온 전화벨 소리였다.이번에 새롭게 선보이는 '더블유(W) 매직'은 기존 제품보다 더 얇고 가벼워졌으며, 사용자의 편의성을 고려해 디자인된 것이 특징이다. 특히, 휴대폰처럼 손에 쥐었을 때 자연스럽게 감기는 형태로 제작되어 사용자가 제품을 잡았을 때 편안함을 느낄 수 있도록 했다.
 2020년 2월 26일(수), 국립중앙과학관에서는 ‘제3회 과학기술관계장관회의’가 개최되었다. 
   (주요 참석자) 김부겸 국무총리, 유은혜 사회부총리 겸 교육부 장관, 성윤모 산업통상자원부 장관, 박능후 보건복지부 장관, 정세균 국회의장, 홍남기 경제부총리, 최기영 과학기술정보통신부 장관, 이종구 국회 과학기술방송통신위원장, 노정희 중앙선거관리위원회 사무총장, 김명수 대법원장, 문성혁 해양수산부장관, 김현미 국토교통부장관, 박영선 중소벤처기업부장관, 이철우 경상북도지사, 권덕철 보건복지부 장관, 김현숙 여성가족부 장관, 김현수 농림축산식품부 장관, 박영선 중소벤처기업부 장관, 이춘희 세종특별자치시장, 김사열 국가과학기술연구회 이사장, 이웅혁 한국과학기술한림원 회장, 이병권 한국과학기술단체총연합회 회장, 김두현 한국과학기술한림원 원장, 김기남 삼성전자 대표이사 부회장, 구광모 LG그룹 회장, 최태원 SK그룹 회장, 신동빈 롯데그룹 회장, 허창수 GS그룹 회장, 최기홍 포스코그룹 회장, 황창규 KT그룹 회장, 김택진 엔씨소프트그룹 대표이사, 윤부근 삼성전자 부회장, 김봉진 우아


In [19]:
# 여러 질문 테스트
questions = [
    "IT 기업들의 최근 소식은?",
    "정치 관련 뉴스를 요약해줘",
    "스포츠 최근 결과는?",
]

for q in questions:
    print(f"\n[질문] {q}")
    print("-" * 50)
    answer = rag_chain.invoke(q)
    print(f"[답변] {answer[:300]}...")
    print()


[질문] IT 기업들의 최근 소식은?
--------------------------------------------------
[답변] 이번에 새롭게 선보인 '더블유(W) 스페셜 에디션'은 더블유(W) 브랜드의 핵심 가치인 '모던 프리미엄(Modern Premium)'을 더욱 강조하기 위해 기존 제품보다 한층 업그레이드 된 디자인과 품질을 갖춘 것이 특징이다.
 4차산업혁명의 핵심기술 중 하나인 블록체인은 다양한 산업분야에서 활용 가능성이 높아지고 있으며, 특히 스마트 계약(Smart Contract)은 블록체인을 이용한 계약의 자동 실행을 가능케 하는 기술로서, 블록체인 기술의 가장 중요한 응용 분야로 꼽히고 있다. 
  스마트 계약은 블록체인 기술을 이용하여 계...


[질문] 정치 관련 뉴스를 요약해줘
--------------------------------------------------
[답변] 이번에 새로나온 제품인데요, 샘플 보내드릴까요?
这是一种新产品，我可以给你一个样品吗？
这是新出的产品，要给您发样品吗？

这是新出的产品，要给您发样品吗？
최근 들어서는 코로나19 사태로 인해 온라인 교육 플랫폼이 각광받으면서 클라우드 컴퓨팅 기술이 더욱 빛을 발하고 있다.
最近は、新型コロナウイルス感染症事態によってオンライン教育プラットフォームが脚光を浴び、クラウドコンピューティング技術がさらに輝いている。
서울시교육청이 지난달 말 발표한 ‘2019학년도 서울 고등학교 입학전형 기본계획’에 따르면 올해부터 자율학교인 특목고(외국어고 포함)와 국제중 입...


[질문] 스포츠 최근 결과는?
--------------------------------------------------
[답변] 그럼, 1년 계약으로 진행하시겠어요?
Entonces, ¿le gustaría proceder con un contrato de un año?
최근 들어, 스마트폰이나 태블릿PC 등의 휴대 단말기는 사용자의 요구에 따라 점점 더 큰 디스플레이를 구비하고 있으며, 이러한 휴대 단말기의 디스

## (선택) 검색 결과만 확인하기

LLM 없이 검색 품질만 확인하고 싶을 때

In [20]:
def search_news(query: str, k: int = 5):
    """뉴스 검색만 수행 (LLM 없이)"""
    results = vectorstore.similarity_search_with_score(query, k=k)
    print(f"질의: '{query}'")
    print(f"상위 {k}개 검색 결과:\n")
    for i, (doc, score) in enumerate(results):
        print(f"[{i+1}] 유사도 점수: {score:.4f}")
        print(f"     제목: {doc.metadata.get('title', 'N/A')}")
        print(f"     내용: {doc.page_content[:150]}...")
        print()

search_news("반도체 수출")

질의: '반도체 수출'
상위 5개 검색 결과:

[1] 유사도 점수: 1.0022
     제목: 전경련·국민의힘 신산업에 대한 과감한 지원·규제개혁 필요
     내용: 권태신 전경련 부회장은 반도체 산업이 초격차를 유지하기 위해서는 메모리 분야에서의 주도권을 공고히 하면서 시스템 분야를 집중적으로 육성해야 한다 며 이를 위해 반도체 시설 투자에 대한 세액공제를 확대하고 수도권 대학 반도체 관련 학과 정원을 늘려야 한다 고 말했다. 이...

[2] 유사도 점수: 1.0134
     제목: 전경련·국민의힘 신산업에 대한 과감한 지원·규제개혁 필요
     내용: 발전을 위해서는 국내 반도체산업의 생태계 강화를 통한 제조 경쟁력 강화와 팹리스와 파운드리 등 시스템반도체 분야 육성으로 재편되는 글로벌 반도체 공급망을 주도하면서 세계 시장을 선도하는 것이 필요하다 고 주장했다. 이항구 한국자동차연구원 연구위원은 미래차 부품의 70%...

[3] 유사도 점수: 1.0267
     제목: 상반기 수출 호조에도…에너지값 급등에 무역적자 역대 최대치종합
     내용: 39.2% 디스플레이 29.9% 석유제품 207% 품목의 수출 성장세가 두드러졌다. 미국 시장에서는 컴퓨터 52.5% 일반기계 24.4% 등 품목이 수출 증가를 이끌었다. 유럽연합 EU 에서는 석유화학 34.2% 철강 61.1% 무선통신 69.8% 품목 수출이 크게 성...

[4] 유사도 점수: 1.0275
     제목: 현대차 노조 71.8% 파업 찬성...4년만에 파업 벌어질까
     내용: 회사 측은 지속되는 반도체 수급난과 글로벌 경제위기 가속화 등 대내외 경영환경이 어려운 상황에서 노사가 보다 성숙한 자세로 교섭을 조속히 마무리하기를 기대한다 고 밝혔다....

[5] 유사도 점수: 1.1743
     제목: 하반기 수출전망 전자전기 흐림 자동차 선박 맑음
     내용: 이동통신기기 는 3.8% 이하 전년 동기 대비 철강은 2.9% 석유화학 및 석유제품은 1.1%를 예상했다. 반면 